# Renewables in Electricity Markets - Assignment 1
### Course 46755, DTU Wind (Technical University of Denmark) - Instructor: Jalal Kazempour

This notebook implements **Step 1: Copper-Plate, Single Hour** of Assignment 1.

**Case study:** IEEE 24-bus Reliability Test System (Table F.4).

**Goal of Step 1:** clear a single-hour, single-node ("copper-plate") electricity market with price-elastic demand, and determine:

1. The market-clearing price under a uniform pricing scheme
2. The social welfare of the system
3. The profit of each producer, including both conventional units and wind farms
4. The utility of each demand
5. Verification of the market-clearing price via the KKT conditions


## 1. Imports and input data

The input data (conventional generators, wind farms, demands, transmission lines)
was transcribed from the IEEE 24-bus Reliability Test System technical data table
into `data/Assignment1_IEEE24bus_input_data.xlsx`.


In [1]:
import pandas as pd
import numpy as np
import os
from scipy.optimize import linprog


In [2]:
# From Excel: Assignment 1, IEEE 24-bus input data
data_path = os.path.join("data", "Assignment1_IEEE24bus_input_data.xlsx")
data_file = pd.read_excel(data_path, sheet_name=None)  # returns all sheets as a dict

gens = data_file["Conventional_generators"]
wind = data_file["Wind_farms"]
demands = data_file["Demands"]
lines = data_file["Transmission_lines"]

gens


,Generator,Location_node,Production_cost_USD_per_MWh,Upward_reserve_cost_USD_per_MW,Downward_reserve_cost_USD_per_MW,Capacity_MW,Max_upward_reserve_MW,Max_downward_reserve_MW
0,G1,1,13.32,1.68,2.32,106.4,48,48
1,G2,2,13.32,1.68,2.32,106.4,48,48
2,G3,7,20.70,3.30,4.67,245.0,84,84
3,G4,13,20.93,4.07,3.93,413.7,216,216
4,G5,15,26.11,1.89,3.11,42.0,42,42
5,G6,15,10.52,5.48,3.52,108.5,36,36
6,G7,16,10.52,5.48,3.52,108.5,36,36
7,G8,18,6.02,4.98,5.02,280.0,60,60
8,G9,21,5.47,5.53,4.97,280.0,60,60
9,G10,22,7.00,8.00,6.00,210.0,48,48


## 2. Market-clearing price (copper-plate, single hour)

**Model:** single node ("copper-plate", i.e. the transmission network is ignored),
single hour, price-elastic demand.

**Optimization problem** (Lecture 2, slides 85-89):

$$
\text{Maximize} \quad SW = \sum_{d} U_d \, p_d \; - \; \sum_{g} C_g \, p_g
$$

$$
\text{subject to:} \qquad
0 \le p_g \le \bar{P}_g \quad \forall g
\qquad\qquad
0 \le p_d \le \bar{P}_d \quad \forall d
$$

$$
\sum_{d} p_d \; - \; \sum_{g} p_g \; = \; 0 \;:\; \lambda
$$

where:
- $U_d$ = bid price of demand $d$ [\$/MWh]
- $C_g$ = offer price of producer $g$ [\$/MWh] (conventional generators and wind farms)
- $\bar{P}_g$ = capacity of producer $g$ (or day-ahead forecast for wind) [MW]
- $\bar{P}_d$ = maximum load of demand $d$ [MW]
- $\lambda$ = dual variable of the power-balance constraint = **market-clearing price**

$SW$ is the **social welfare**: the total value created by the market, equal to the
sum of what consumers are willing to pay for the energy they receive, minus the
total cost incurred by producers to generate it. Maximizing $SW$ subject to the
physical/technical limits of every unit is, by the first welfare theorem applied
to this idealized (perfectly competitive, no market power) setting, equivalent to
what a competitive uniform-price auction achieves.


### 2.1 Build the supply side (offer curve)

Producers = conventional generators (offer price = production cost) + wind farms
(offer price = \$0/MWh, quantity = day-ahead forecast, as stated in the assignment).
Wind is offered at zero cost because, once built, producing wind energy has (near)
zero marginal cost - this is what places it first in the merit order.


In [3]:
supplier_names = list(gens["Generator"]) + list(wind["Wind_farm"])
supplier_cost = np.concatenate([
    gens["Production_cost_USD_per_MWh"].to_numpy(),
    np.zeros(len(wind)),
])
supplier_capacity = np.concatenate([
    gens["Capacity_MW"].to_numpy(),
    wind["Day_ahead_forecast_MW"].to_numpy(),
])
n_sup = len(supplier_names)

pd.DataFrame({"supplier": supplier_names, "offer_price": supplier_cost, "capacity_MW": supplier_capacity})


,supplier,offer_price,capacity_MW
0,G1,13.32,106.40
1,G2,13.32,106.40
2,G3,20.70,245.00
3,G4,20.93,413.70
4,G5,26.11,42.00
5,G6,10.52,108.50
6,G7,10.52,108.50
7,G8,6.02,280.00
8,G9,5.47,280.00
9,G10,7.00,210.00


### 2.2 Build the demand side (bid curve)

**Assumption:** the demand bid price is set equal to the curtailment cost
(500 \$/MWh), which is much higher than any generation offer price, as requested
by the assignment ("use comparatively high values relative to the generation
cost of conventional units, to ensure that most demands are supplied"). Economically,
the curtailment cost represents the *Value of Lost Load* (VoLL): the maximum price a
consumer would rationally accept to pay rather than being disconnected, which is
exactly the definition of a demand bid price in an elastic-demand market model.


In [4]:
demand_names = list(demands["Demand"])
demand_bid = demands["Curtailment_cost_USD_per_MWh"].to_numpy()
demand_maxload = demands["Consumption_MW"].to_numpy()
n_dem = len(demand_names)

pd.DataFrame({"demand": demand_names, "bid_price": demand_bid, "max_load_MW": demand_maxload})


,demand,bid_price,max_load_MW
0,D1,500,84
1,D2,500,75
2,D3,500,139
3,D4,500,58
4,D5,500,55
5,D6,500,106
6,D7,500,97
7,D8,500,132
8,D9,500,135
9,D10,500,150


### 2.3 Linear programming formulation

`scipy.optimize.linprog` only **minimizes**, so we minimize $-SW$ instead of
maximizing $SW$:

$$
\text{minimize} \quad \sum_{g} C_g \, p_g \; - \; \sum_{d} U_d \, p_d
$$

The decision vector is $x = [\, p_{sup} \;(n_{sup}) \, , \; p_{dem} \;(n_{dem}) \,]$.
This is a linear program (LP): the objective and every constraint are linear in
$x$, and the feasible region (a box intersected with one hyperplane) is convex,
so any local optimum found is guaranteed to be the **global** optimum.


In [5]:
# Objective vector: minimize sum(C_g * p_g) - sum(U_d * p_d)
c = np.concatenate([supplier_cost, -demand_bid])

# Bounds: 0 <= p_sup <= capacity , 0 <= p_dem <= max load
bounds = [(0, cap) for cap in supplier_capacity] + [(0, load) for load in demand_maxload]

# Power balance equality constraint: sum(p_dem) - sum(p_sup) = 0
A_eq = np.concatenate([-np.ones(n_sup), np.ones(n_dem)]).reshape(1, -1)
b_eq = np.array([0.0])


In [6]:
result = linprog(c=c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")

if not result.success:
    raise RuntimeError(f"Market-clearing LP did not solve: {result.message}")

p_sup_opt = result.x[:n_sup]
p_dem_opt = result.x[n_sup:]

dispatch_gens = pd.Series(p_sup_opt, index=supplier_names, name="Dispatch_MW").round(2)
dispatch_dem = pd.Series(p_dem_opt, index=demand_names, name="Consumption_MW").round(2)

print("Dispatch of producers (MW):")
print(dispatch_gens)
print("\nDispatch of demands (MW):")
print(dispatch_dem)


Dispatch of producers (MW):
G1     106.40
G2     106.40
G3     217.64
G4       0.00
G5       0.00
G6     108.50
G7     108.50
G8     280.00
G9     280.00
G10    210.00
G11    217.00
G12    245.00
W1     120.54
W2     115.52
W3      53.34
W4      38.16
Name: Dispatch_MW, dtype: float64

Dispatch of demands (MW):
D1      84.0
D2      75.0
D3     139.0
D4      58.0
D5      55.0
D6     106.0
D7      97.0
D8     132.0
D9     135.0
D10    150.0
D11    205.0
D12    150.0
D13    245.0
D14     77.0
D15    258.0
D16    141.0
D17    100.0
Name: Consumption_MW, dtype: float64


### 2.4 Market-clearing price

We compute the price in two independent ways, to cross-check the result:

- **Method A:** the dual variable (Lagrange multiplier) of the power-balance
  constraint. Since we minimized $f = -SW$ subject to
  $\big(\sum_d p_d - \sum_{sup} p_{sup}\big) = 0$, we have
  $\dfrac{d(\min f)}{d(b_{eq})} = -\dfrac{d(SW^*)}{d(b_{eq})} = -\lambda
  \;\Rightarrow\; \lambda = -\text{marginal}$.

- **Method B (economic / KKT intuition, Lecture 2 slides 73-77):** the market-clearing
  price equals the offer price of the **marginal producer** - the price-setting
  unit dispatched strictly between 0 and its capacity (neither bound is binding,
  so both of its capacity multipliers are zero by complementary slackness). Section
  4 below derives and checks this rigorously for *every* unit, not just the marginal one.


In [7]:
social_welfare = -result.fun  # result.fun = min(-SW) = -SW*, so SW* = -result.fun

# Method A: dual variable of the power-balance constraint
price_from_dual = -result.eqlin.marginals[0]

# Method B: identify the marginal (price-setting) producer
tol = 1e-6
is_marginal = (p_sup_opt > tol) & (p_sup_opt < supplier_capacity - tol)
marginal_unit = np.array(supplier_names)[is_marginal]
marginal_price = supplier_cost[is_marginal]

print(f"Total social welfare: {social_welfare:,.2f} $")
print(f"Market-clearing price (Method A, dual variable): {price_from_dual:.2f} $/MWh")
print(f"Marginal producer (Method B, KKT check): {marginal_unit} at {marginal_price} $/MWh")


Total social welfare: 1,084,239.43 $
Market-clearing price (Method A, dual variable): 20.70 $/MWh
Marginal producer (Method B, KKT check): ['G3'] at [20.7] $/MWh


## 3. Social welfare, producer profit and demand utility

Once the market-clearing price $\lambda$ and the optimal dispatch
$(p_g^\star, p_d^\star)$ are known, every other economic outcome follows directly.

**Profit of producer $g$** (uniform pricing: every accepted MW is paid the same
price $\lambda$, regardless of the producer's own offer price):

$$
\pi_g \;=\; \underbrace{\lambda \, p_g^\star}_{\text{revenue}} \; - \;
\underbrace{C_g \, p_g^\star}_{\text{cost}} \;=\; (\lambda - C_g)\, p_g^\star
$$

A producer only earns a strictly positive profit if it is *inframarginal*, i.e. if
its offer price $C_g$ is strictly below the market price $\lambda$. Wind farms
(offer price \$0/MWh) are always inframarginal whenever $\lambda > 0$, so they earn
the full market price on every MWh produced - this is the essence of the
**merit-order effect**: cheap/zero-marginal-cost renewables suppress the price and
capture it as pure profit (an "infra-marginal rent").

**Utility of demand $d$** (as defined in the assignment):

$$
u_d \;=\; p_d^\star \, \big( U_d \, - \, \lambda \big)
$$

This is the consumer-side analogue of producer profit: a demand only benefits if
its bid price $U_d$ (its willingness to pay) exceeds the price it actually pays,
$\lambda$.

**Consistency check.** Because the market-clearing problem maximizes
$SW = \sum_d U_d p_d - \sum_g C_g p_g$, and $\lambda \sum_d p_d = \lambda \sum_g p_g$
at the optimum (power balance), the following identity must hold exactly:

$$
SW \;=\; \sum_g \pi_g \; + \; \sum_d u_d
$$

i.e. the social welfare "pie" is split, without any leftover, between producer
profits and consumer utilities. We verify this numerically below.


In [8]:
# Profit of producer g = revenue - cost = (price - offer_price) * dispatched_quantity
producer_profit = (price_from_dual - supplier_cost) * p_sup_opt

# Utility of demand d = consumption * (bid_price - price)
demand_utility = p_dem_opt * (demand_bid - price_from_dual)

producers_summary = pd.DataFrame({
    "Offer_price_USD_per_MWh": supplier_cost,
    "Dispatch_MW": p_sup_opt,
    "Profit_USD": producer_profit,
}, index=supplier_names).round(2)

demands_summary = pd.DataFrame({
    "Bid_price_USD_per_MWh": demand_bid,
    "Consumption_MW": p_dem_opt,
    "Utility_USD": demand_utility,
}, index=demand_names).round(2)

print("Producers: offer price, dispatch and profit")
print(producers_summary)
print("\nDemands: bid price, consumption and utility")
print(demands_summary)


Producers: offer price, dispatch and profit
     Offer_price_USD_per_MWh  Dispatch_MW  Profit_USD
G1                     13.32       106.40      785.23
G2                     13.32       106.40      785.23
G3                     20.70       217.64        0.00
G4                     20.93         0.00       -0.00
G5                     26.11         0.00       -0.00
G6                     10.52       108.50     1104.53
G7                     10.52       108.50     1104.53
G8                      6.02       280.00     4110.40
G9                      5.47       280.00     4264.40
G10                     7.00       210.00     2877.00
G11                    10.52       217.00     2209.06
G12                    10.89       245.00     2403.45
W1                      0.00       120.54     2495.18
W2                      0.00       115.52     2391.26
W3                      0.00        53.34     1104.14
W4                      0.00        38.16      789.91

Demands: bid price, consumption and u

In [9]:
total_profit = producer_profit.sum()
total_utility = demand_utility.sum()

print(f"Sum of producer profits : {total_profit:,.2f} $")
print(f"Sum of demand utilities : {total_utility:,.2f} $")
print(f"Sum of both             : {total_profit + total_utility:,.2f} $")
print(f"Social welfare (SW*)    : {social_welfare:,.2f} $")


Sum of producer profits : 26,424.33 $
Sum of demand utilities : 1,057,815.10 $
Sum of both             : 1,084,239.43 $
Social welfare (SW*)    : 1,084,239.43 $


**Observations:**

- Wind farms (W1-W4) earn the highest profits *per MWh* of any unit, since their
  cost is \$0/MWh yet they are paid the full market price of 20.70 \$/MWh -
  a direct illustration of the merit-order effect benefiting low-marginal-cost
  renewables.
- The marginal generator (G3) earns **zero profit**: it is paid exactly its own
  offer price (20.70 \$/MWh), so revenue equals cost. This is a general property
  of uniform-price auctions: the price-setting unit never profits from the auction
  itself.
- G4 and G5 are not dispatched (offer price above the market price) and correctly
  show zero profit.
- All 17 demands are served in full (bid price 500 \$/MWh $\gg$ 20.70 \$/MWh), so
  every demand's utility is strictly positive and proportional to its consumption.


## 4. Verifying the market-clearing price via the KKT conditions

The assignment explicitly asks us to **verify** the market-clearing price using
the Karush-Kuhn-Tucker (KKT) conditions, rather than just trusting the LP solver's
output. This section rebuilds the Lagrangian of the minimization problem solved in
Section 2.3 and checks its optimality conditions directly.

**Lagrangian** (Lecture 2, slides 61-71). For our problem,
$\;\text{minimize } f(x)=\sum_g C_g p_g - \sum_d U_d p_d\;$ subject to
$-p_g\le 0$, $p_g-\bar P_g\le 0$, $-p_d\le 0$, $p_d-\bar P_d\le 0$, and
$\sum_d p_d - \sum_g p_g = 0$:

$$
\mathcal{L} = \sum_g C_g p_g - \sum_d U_d p_d
+ \lambda\Big(\sum_d p_d - \sum_g p_g\Big)
+ \sum_g\big[-p_g\,\underline{\mu}_g + (p_g-\bar P_g)\,\overline{\mu}_g\big]
+ \sum_d\big[-p_d\,\underline{\mu}_d + (p_d-\bar P_d)\,\overline{\mu}_d\big]
$$

**Stationarity** ($\partial\mathcal{L}/\partial p_g = 0$ and
$\partial\mathcal{L}/\partial p_d = 0$) gives, for every producer $g$ and demand $d$:

$$
\lambda = C_g - \underline{\mu}_g + \overline{\mu}_g
\qquad\qquad
\lambda = U_d + \underline{\mu}_d - \overline{\mu}_d
$$

**Complementary slackness** requires
$\underline{\mu}_g \, p_g = 0$, $\overline{\mu}_g\,(\bar P_g - p_g) = 0$ (and
likewise for demands), together with dual feasibility
$\underline{\mu}_g,\overline{\mu}_g,\underline{\mu}_d,\overline{\mu}_d \ge 0$.

Because a unit cannot simultaneously be at its lower bound ($p_g=0$) and its upper
bound ($p_g=\bar P_g$) unless $\bar P_g=0$, **at most one** of
$\underline{\mu}_g,\overline{\mu}_g$ can be non-zero. Solving the two
stationarity equations above under this restriction gives closed-form expressions
for the implied multipliers:

$$
\overline{\mu}_g = \max(\lambda - C_g,\, 0), \qquad
\underline{\mu}_g = \max(C_g - \lambda,\, 0)
$$

$$
\overline{\mu}_d = \max(U_d - \lambda,\, 0), \qquad
\underline{\mu}_d = \max(\lambda - U_d,\, 0)
$$

These formulas already guarantee dual feasibility ($\mu \ge 0$ by construction, since
they are defined as a $\max$ with 0). What is **not** automatic is complementary
slackness: we must check numerically that these implied multipliers are only
non-zero when the corresponding bound is actually active. That is what the code
below verifies for every single producer and demand.


In [10]:
# Implied dual variables (only one of each pair can be non-zero, by construction)
mu_sup_up = np.maximum(price_from_dual - supplier_cost, 0.0)   # positive => should be at capacity
mu_sup_low = np.maximum(supplier_cost - price_from_dual, 0.0)  # positive => should be at zero
mu_dem_up = np.maximum(demand_bid - price_from_dual, 0.0)      # positive => should be fully served
mu_dem_low = np.maximum(price_from_dual - demand_bid, 0.0)     # positive => should be at zero

# Complementary-slackness residuals: mu_low * p and mu_up * (Pbar - p) must both be ~0
cs_sup = np.maximum(mu_sup_low * p_sup_opt, mu_sup_up * (supplier_capacity - p_sup_opt))
cs_dem = np.maximum(mu_dem_low * p_dem_opt, mu_dem_up * (demand_maxload - p_dem_opt))

kkt_table = pd.DataFrame({
    "cost_or_bid": np.concatenate([supplier_cost, demand_bid]),
    "dispatch_MW": np.concatenate([p_sup_opt, p_dem_opt]),
    "mu_low": np.concatenate([mu_sup_low, mu_dem_low]),
    "mu_up": np.concatenate([mu_sup_up, mu_dem_up]),
    "complementary_slackness_residual": np.concatenate([cs_sup, cs_dem]),
}, index=supplier_names + demand_names).round(6)

kkt_table


,cost_or_bid,dispatch_MW,mu_low,mu_up,complementary_slackness_residual
G1,13.32,106.40,0.00,7.38,0.0
G2,13.32,106.40,0.00,7.38,0.0
G3,20.70,217.64,0.00,0.00,0.0
G4,20.93,0.00,0.23,0.00,0.0
G5,26.11,0.00,5.41,0.00,0.0
G6,10.52,108.50,0.00,10.18,0.0
G7,10.52,108.50,0.00,10.18,0.0
G8,6.02,280.00,0.00,14.68,0.0
G9,5.47,280.00,0.00,15.23,0.0
G10,7.00,210.00,0.00,13.70,0.0


In [11]:
cs_tol = 1e-3  # numerical tolerance (LP solver precision)
kkt_verified = bool(kkt_table["complementary_slackness_residual"].max() < cs_tol)

print(f"Max complementary-slackness residual: {kkt_table['complementary_slackness_residual'].max():.6f}")
print(f"KKT conditions verified for lambda = {price_from_dual:.2f} $/MWh: {kkt_verified}")


Max complementary-slackness residual: 0.000000
KKT conditions verified for lambda = 20.70 $/MWh: True


**Reading the table:** every unit falls into exactly one of three categories,
consistent with economic intuition:

| Situation | Condition | Implied multipliers |
|---|---|---|
| Not dispatched at all ($p=0$) | offer price / bid price on the "wrong side" of $\lambda$ | one multiplier $>0$, the other $=0$ |
| Fully dispatched ($p=\bar P$) | offer price / bid price on the "right side" of $\lambda$ | one multiplier $>0$, the other $=0$ |
| Marginal (price-setting) unit, $0<p<\bar P$ | offer price $=\lambda$ (or bid price $=\lambda$) | both multipliers $=0$ |

Since every complementary-slackness residual is (numerically) zero, the KKT
conditions confirm - independently of the LP solver's internal dual output - that
**$\lambda = 20.70$ \$/MWh is indeed the market-clearing price**.


## 5. Summary of Step 1 results

| Quantity | Value |
|---|---|
| Market-clearing price | **20.70 \$/MWh** |
| Marginal (price-setting) producer | **G3** (node 7) |
| Total social welfare | **1,084,239.43 \$** |
| Sum of producer profits | see Section 3 |
| Sum of demand utilities | see Section 3 |
| KKT conditions | **verified** (residual < 1e-3) |

Total generation available (conventional + wind, 2690.06 MW) exceeds total demand
(2207 MW), and since every demand bid price (500 \$/MWh) is far above every
generator's offer price, **all 17 demands are served in full** at the optimum, and
generation is dispatched strictly in merit order (cheapest first) until the 2207 MW
of demand is met.
